In [ ]:
from pathlib import Path
import subprocess
import time
import requests

PROJECT_ROOT = Path(r"X:\dev\projects\study\ATLAS")
LLAMA_DIR = PROJECT_ROOT / "llama"
MODEL_PATH = PROJECT_ROOT / "models" / "gemma3" / "gemma-3-4b-it-Q4_K_M.gguf"
LOG_PATH = PROJECT_ROOT / "rag" / "llama_server.log"

HOST = "127.0.0.1"
PORT = 8080

In [ ]:
SERVER_CMD = [
    str(LLAMA_DIR / "llama-server.exe"),
    "-m", str(MODEL_PATH),
    "-ngl", "999",
    "-c", "8192",
    "--host", HOST,
    "--port", str(PORT),
]

log_f = open(LOG_PATH, "w", encoding="utf-8")
llama_proc = subprocess.Popen(
    SERVER_CMD,
    cwd=str(LLAMA_DIR),
    stdout=log_f,
    stderr=subprocess.STDOUT,
    creationflags=subprocess.CREATE_NEW_PROCESS_GROUP,
)
print(f"llama-server PID={llama_proc.pid}")

for _ in range(30):
    try:
        requests.get(f"http://{HOST}:{PORT}/health", timeout=2)
        print("server ready")
        break
    except Exception:
        time.sleep(2)
else:
    print("server did not start in time, check the log")

In [ ]:
import sys
sys.path.insert(0, str(PROJECT_ROOT))

from rag.search import load_index, rag_answer

idx = load_index(PROJECT_ROOT / "data" / "index")

In [ ]:
question = "what do the 3 dots mean in math"
answer, sources = rag_answer(question, idx)

print(f"Q: {question}")
print(f"A: {answer}")
print()
for s in sources:
    print(f"  [{s['title']}] chunk {s['chunk_idx']}")

In [ ]:
question = "who wrote the song photograph by ringo starr"
answer, sources = rag_answer(question, idx)

print(f"Q: {question}")
print(f"A: {answer}")
print()
for s in sources:
    print(f"  [{s['title']}] chunk {s['chunk_idx']}")

In [ ]:
import gc

if llama_proc.poll() is None:
    llama_proc.terminate()
    try:
        llama_proc.wait(timeout=5)
    except subprocess.TimeoutExpired:
        llama_proc.kill()
        llama_proc.wait()

log_f.close()

idx["qdrant"].close()
del idx
gc.collect()

print(f"server stopped (exit code {llama_proc.returncode})")